# ARIMA Diagnostic Workflow

Module: Financial Time Series

## Lesson summary

This lab builds a reproducible ARIMA workflow. Students start with a price-like process, transform it into returns, test stationarity, search over small ARIMA orders, and validate residuals {cite}`box2015time,hamilton1994time`.

## Learning objectives

By the end of this lab, students should be able to:

- explain why raw prices and returns usually require different models;
- use the ADF test as a stationarity diagnostic;
- compare ARIMA candidates with AIC and BIC;
- run residual autocorrelation and normality checks;
- write a model-selection conclusion that includes limitations.

## Model equations

An ARIMA model applies autoregressive and moving-average structure after differencing:

$$
\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t,
$$

where $\phi(L)$ and $\theta(L)$ are lag polynomials. In this lab, the return series is modeled with $d=0$, and the residual diagnostics ask whether the fitted innovations behave like:

$$
\mathbb{E}[\varepsilon_t] = 0, \qquad \operatorname{Cov}(\varepsilon_t,\varepsilon_{t-k}) = 0 \quad \text{for } k>0.
$$

## Setup

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

from src.market_data_quality import log_returns
from src.time_series_diagnostics import (
    adf_report,
    arima_order_search,
    jarque_bera_report,
    ljung_box_report,
)

## Simulated price and return data

In [ ]:
rng = np.random.default_rng(7)
dates = pd.bdate_range("2023-01-02", periods=500)

innovations = rng.normal(0, 0.01, len(dates))
log_price = np.cumsum(0.0002 + innovations)
prices = pd.Series(100 * np.exp(log_price), index=dates, name="synthetic_price")
returns = log_returns(prices).rename("log_return")

pd.DataFrame({"price": prices, "log_return": returns}).head()

## Stationarity check

In [ ]:
pd.DataFrame(
    {
        "price_level": adf_report(prices),
        "log_return": adf_report(returns),
    }
)

## ARIMA order search

The grid is intentionally small for classroom use. Candidate ranking uses information criteria in the Akaike and Schwarz/BIC tradition {cite}`akaike1974new,schwarz1978estimating`.

In [ ]:
candidate_results = arima_order_search(
    returns,
    p_values=range(0, 4),
    d_values=[0],
    q_values=range(0, 4),
)

candidate_results.head(10)

## Fit the selected model

In [ ]:
best = candidate_results.dropna(subset=["aic"]).iloc[0]
order = (int(best["p"]), int(best["d"]), int(best["q"]))
order

In [ ]:
model = ARIMA(
    returns,
    order=order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
result = model.fit()
result.summary()

## Residual diagnostics

In [ ]:
residuals = pd.Series(result.resid, index=returns.index, name="residuals")

ljung_box_report(
    residuals,
    lags=[5, 10, 20],
    model_df=order[0] + order[2],
)

In [ ]:
jarque_bera_report(residuals)

## Interpretation checklist

| Diagnostic | What to write |
| --- | --- |
| ADF on prices | whether the level appears non-stationary |
| ADF on returns | whether returns are a more plausible modeling target |
| AIC/BIC | which model balances fit and complexity |
| Ljung-Box | whether residual linear autocorrelation remains |
| Jarque-Bera | whether Gaussian residual assumptions are plausible |
| Limitation | why this model is not a forecasting guarantee |

## Model limitations

- ARIMA models summarize linear dependence and can miss regime changes, volatility clustering, and nonlinear dynamics.
- Information criteria compare candidate specifications, but they do not prove that the chosen model is economically meaningful.
- Forecast quality depends on the stationarity transformation, lag search range, and stability of the sample period.